# Spotify Özellik Mühendisliği ve Model Hazırlığı
Bu notebook, Spotify veri setini kullanarak makine öğrenmesi modelleri için özellik mühendisliği (feature engineering) ve veri hazırlama süreçlerini içerir.

## 1. Spark Oturumunu Başlatma
Delta Lake desteğiyle birlikte Spark oturumunu konfigüre ediyoruz.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, size, split
from pyspark.ml.feature import StringIndexer, VectorAssembler

spark = (
    SparkSession.builder
    .appName("Spotify Feature Engineering")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/spark-warehouse")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark oturumu başlatıldı.")

Spark oturumu başlatıldı.


## 2. Silver Tablosunu Okuma
Temizlenmiş verilerin bulunduğu Silver katmanından veriyi çekiyoruz.

In [2]:
SILVER_PATH = "/home/jovyan/work/delta/silver/spotify_tracks"
df = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver katmanından {df.count()} kayıt okundu.")
df.show(5)

Silver katmanından 24673 kayıt okundu.
+--------------------+--------------------+---------+------+--------------------+------------------+---------+--------------------+---------------+--------------------+--------------------+-----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+--------------------+--------------------+------------+---------------------+
|            raw_json|     kafka_timestamp|partition|offset|          event_time|        event_type|  user_id|            track_id|        artists|          album_name|          track_name|track_genre|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|      ingestion_time|     event_timestamp|explicit_int|silver_ingestion_time|
+--------------------+--------------------+---------+------+--------------------+------------------

## 3. Özellik Mühendisliği (Feature Engineering)
Modelin tahmin gücünü artırmak için 5 yeni özellik ekliyoruz:
- **artist_count**: Şarkıdaki sanatçı sayısı.
- **is_short_track**: 2.5 dakikadan kısa olan şarkılar (Trend takibi).
- **energy_dance_interaction**: Enerji ve dans edilebilirlik etkileşimi.
- **mood_index**: Valence ve Enerji çarpımı ile coşku ölçümü.
- **is_feat**: Şarkı isminde düet (ft./feat) bilgisi.

In [3]:
# Özellik 1: Sanatçı Sayısı
df_fe = df.withColumn("artist_count", size(split(col("artists"), ";")))

# Özellik 2: Kısa Şarkı Trendi (150.000 ms)
df_fe = df_fe.withColumn("is_short_track", when(col("duration_ms") < 150000, 1).otherwise(0))

# Özellik 3: Parti Çapraz Etkileşimi
df_fe = df_fe.withColumn("energy_dance_interaction", col("energy") * col("danceability"))

# Özellik 4: Ruh Hali İndeksi
df_fe = df_fe.withColumn("mood_index", col("valence") * col("energy"))

# Özellik 5: Düet/Katılım Durumu
df_fe = df_fe.withColumn("is_feat", when(col("track_name").rlike(r"(?i)\b(feat|ft)\b"), 1).otherwise(0))

print("Yeni özellikler başarıyla eklendi.")

Yeni özellikler başarıyla eklendi.


## 4. Kategorik Değişkenleri Sayısallaştırma
`track_genre` sütununu ML algoritmalarının anlayabileceği index formatına çeviriyoruz.

In [4]:
indexer = StringIndexer(inputCol="track_genre", outputCol="track_genre_index", handleInvalid="keep")
df_fe = indexer.fit(df_fe).transform(df_fe)
print("Tür indeksleme tamamlandı.")

Tür indeksleme tamamlandı.


## 5. Özellik Seçimi ve Vektörleştirme
Spark MLlib gereksinimi olarak tüm girdileri tek bir `features` vektöründe topluyoruz.

In [5]:
selected_cols = [
    "track_id", "popularity", "duration_ms", "danceability", "energy", 
    "loudness", "speechiness", "acousticness", "instrumentalness", 
    "liveness", "valence", "tempo", "explicit_int", "track_genre_index",
    "artist_count", "is_short_track", "energy_dance_interaction", "mood_index", "is_feat"
]
df_selected = df_fe.select(selected_cols)

feature_cols = [c for c in selected_cols if c not in ["track_id", "popularity"]]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
df_ml = assembler.transform(df_selected)

print("Vektörleştirme tamamlandı. İlk 5 satır önizlemesi:")
df_ml.select("track_id", "popularity", "features").show(5, truncate=False)

Vektörleştirme tamamlandı. İlk 5 satır önizlemesi:
+----------------------+----------+-------------------------------------------------------------------------------------------------------------------------+
|track_id              |popularity|features                                                                                                                 |
+----------------------+----------+-------------------------------------------------------------------------------------------------------------------------+
|1wJ5Z1Nkcqknyi2U2mjL0G|11        |[892319.0,0.247,0.846,-6.737,0.0531,0.00122,0.586,0.125,0.0698,115.064,0.0,4.0,1.0,0.0,0.20896199999999998,0.0590508,0.0]|
|3leX6DLRl06RD1effNpB0Q|22        |[429540.0,0.424,0.948,-6.401,0.096,2.0E-4,0.00104,0.111,0.134,136.047,0.0,4.0,1.0,0.0,0.401952,0.127032,0.0]             |
|0qZfhc3NDoeWKomjWX3DBU|23        |[220200.0,0.209,0.989,-4.644,0.125,1.87E-6,0.348,0.177,0.113,104.536,0.0,4.0,1.0,0.0,0.206701,0.111757,0.0]              |
|

## 6. Gold Katmanına Kayıt
Model eğitimine hazır veriyi Delta formatında Gold katmanına kaydediyoruz.

In [6]:
GOLD_PATH = "/home/jovyan/work/delta/gold/spotify_tracks_features"
df_ml.write.format("delta").mode("overwrite").save(GOLD_PATH)
print(f"Özellik seti Gold katmanına başarıyla kaydedildi: {GOLD_PATH}")

Özellik seti Gold katmanına başarıyla kaydedildi: /home/jovyan/work/delta/gold/spotify_tracks_features
